In [ ]:
from google.colab import files
uploaded = files.upload()

Saving customer_churn_processed.csv to customer_churn_processed.csv
Saving df_final (1).csv to df_final (1).csv
Saving df_final (2).csv to df_final (2).csv


In [ ]:
import pandas as pd

df = pd.read_csv("customer_churn_processed.csv", parse_dates=["Date"])

# Identify churn date (first date where churn = True per customer)
churn_dates = df[df["Churned"] == True].groupby("CustomerID")["Date"].min()

# Aggregate at customer level
customer_df = df.groupby("CustomerID").agg(
    AvgLogin=("Login", "mean"),
    AvgWatchTime=("WatchTimeMinutes", "mean"),
    FirstLogin=("Date", "min"),
    AgeGroup=("AgeGroup", "first"),
    ContentPreference=("ContentPreference", "first"),
    SubscriptionPlan=("SubscriptionPlan_Encoded", "first"),
    Churn=("Churned", "max")
).reset_index()

# Churn date column
customer_df["ChurnDate"] = customer_df["CustomerID"].map(churn_dates)

# Days to churn
customer_df["DaysToChurn"] = (
    customer_df["ChurnDate"] - customer_df["FirstLogin"]
).dt.days

# If not churned, DaysToChurn as None
customer_df.loc[customer_df["Churn"] == False, "DaysToChurn"] = None

# Churn date if not needed
df1 = customer_df.drop(columns=["ChurnDate"])

# One-hot encoding 'ContentPreference'
df1 = pd.get_dummies(df1, columns=['ContentPreference'], prefix='ContentPreference')

# Converting boolean columns from one-hot encoding to integer (True=1, False=0)
for col in df1.columns:
    if df1[col].dtype == 'bool':
        df1[col] = df1[col].astype(int)

# Replacing NaN values in 'DaysToChurn' with 100
df1['DaysToChurn'] = df1['DaysToChurn'].fillna(100) # 100 implies not churned in this time period
df1.to_csv("customer_churn_transformed.csv", index=False)

print(df1.head())

  CustomerID  AvgLogin  AvgWatchTime FirstLogin AgeGroup  SubscriptionPlan  \
0    CUST001  0.576923     21.423077 2025-04-01    36-50                 0   
1    CUST002  0.505495     17.967033 2025-04-01    26-35                 0   
2    CUST003  0.461538     15.307692 2025-04-01    36-50                 0   
3    CUST004  0.747253     58.549451 2025-04-01      51+                 2   
4    CUST005  0.714286     89.263736 2025-04-01    18-25                 1   

   Churn  DaysToChurn  ContentPreference_Documentaries  \
0      1         51.0                                0   
1      0        100.0                                1   
2      0        100.0                                0   
3      0        100.0                                0   
4      0        100.0                                1   

   ContentPreference_Movies  ContentPreference_Sports  \
0                         0                         1   
1                         0                         0   
2          

In [ ]:
df2 = pd.read_csv("df_final (1).csv")
df3 = pd.read_csv("df_final (2).csv")
print(df1.head(5))
print(df2.head(5))
print(df3.head(5))

df1.rename(columns={df1.columns[0]: "CustomerID"}, inplace=True)
df2.rename(columns={df2.columns[0]: "CustomerID"}, inplace=True)
df3.rename(columns={df3.columns[0]: "CustomerID"}, inplace=True)


  CustomerID  AvgLogin  AvgWatchTime FirstLogin AgeGroup  SubscriptionPlan  \
0    CUST001  0.576923     21.423077 2025-04-01    36-50                 0   
1    CUST002  0.505495     17.967033 2025-04-01    26-35                 0   
2    CUST003  0.461538     15.307692 2025-04-01    36-50                 0   
3    CUST004  0.747253     58.549451 2025-04-01      51+                 2   
4    CUST005  0.714286     89.263736 2025-04-01    18-25                 1   

   Churn  DaysToChurn  ContentPreference_Documentaries  \
0      1         51.0                                0   
1      0        100.0                                1   
2      0        100.0                                0   
3      0        100.0                                0   
4      0        100.0                                1   

   ContentPreference_Movies  ContentPreference_Sports  \
0                         0                         1   
1                         0                         0   
2          

In [ ]:
# Merging all three datasets on CustomerID
df = df1.merge(df2, on="CustomerID", how="inner") \
        .merge(df3, on="CustomerID", how="inner")

df.drop(columns=['SubscriptionPlan'], inplace=True, errors='ignore')

print("Final merged shape:", df.shape)
print(df.head())
print(df.columns)

Final merged shape: (250, 26)
  CustomerID  AvgLogin  AvgWatchTime FirstLogin AgeGroup  Churn  DaysToChurn  \
0    CUST001  0.576923     21.423077 2025-04-01    36-50      1         51.0   
1    CUST002  0.505495     17.967033 2025-04-01    26-35      0        100.0   
2    CUST003  0.461538     15.307692 2025-04-01    36-50      0        100.0   
3    CUST004  0.747253     58.549451 2025-04-01      51+      0        100.0   
4    CUST005  0.714286     89.263736 2025-04-01    18-25      0        100.0   

   ContentPreference_Documentaries  ContentPreference_Movies  \
0                                0                         0   
1                                1                         0   
2                                0                         1   
3                                0                         0   
4                                1                         0   

   ContentPreference_Sports  ...  Location_West  PrimaryDevice_Mobile  \
0                         1  ..

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

X = df.drop(["Churn", "CustomerID"], axis=1, errors="ignore")
y = df["Churn"]

# Drop ChurnDate if it exists
X = X.drop("ChurnDate", axis=1, errors="ignore")

# Converting datetime features to numeric
if "FirstLogin" in X.columns:
    X["FirstLogin"] = pd.to_datetime(X["FirstLogin"], errors="coerce")
    X["DaysSinceFirstLogin"] = (pd.Timestamp.today() - X["FirstLogin"]).dt.days
    X = X.drop("FirstLogin", axis=1)


# One-hot encode categorical variables
X = pd.get_dummies(X, drop_first=True)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)

# Logistic Regression Model
log_reg = LogisticRegression(max_iter=1000, class_weight="balanced")
log_reg.fit(X_train, y_train)

# Evaluation
y_pred = log_reg.predict(X_test)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))



Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.98      0.99        41
           1       0.90      1.00      0.95         9

    accuracy                           0.98        50
   macro avg       0.95      0.99      0.97        50
weighted avg       0.98      0.98      0.98        50



In [ ]:
# Churn probability to each customer using the scaled features used for training
churn_probabilities = log_reg.predict_proba(X_scaled)[:, 1]

# New DataFrame with CustomerID and the calculated churn probabilities
churn_prediction_df = pd.DataFrame({'CustomerID': df['CustomerID'], 'Churn_Probability': churn_probabilities})


# Risk categories
def risk_category(prob):
    if prob < 0.3:
        return "Low Risk"
    elif prob < 0.7:
        return "Medium Risk"
    else:
        return "High Risk"

churn_prediction_df["Risk_Segment"] = churn_prediction_df["Churn_Probability"].apply(risk_category)

print(churn_prediction_df.head())

  CustomerID  Churn_Probability Risk_Segment
0    CUST001           0.998522    High Risk
1    CUST002           0.005313     Low Risk
2    CUST003           0.001887     Low Risk
3    CUST004           0.039369     Low Risk
4    CUST005           0.001925     Low Risk
